In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"online_retail_uk.csv")
print("Raw rows:", df.shape[0] , "Raw columns:", df.shape[1])
df.head()

Raw rows: 495478 Raw columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010 08:26,3.39,17850.0,United Kingdom


EXPLORING THE DATA

In [3]:
print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== NULL VALUES PER COLUMN ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(df.duplicated().sum())

print("\n=== CANCELLED ORDERS (InvoiceNo starting with 'C') ===")
print(df['InvoiceNo'].astype(str).str.startswith('C').sum())

print("\n=== NEGATIVE OR ZERO QUANTITY ===")
print((df['Quantity'] <= 0).sum())

print("\n=== NEGATIVE OR ZERO PRICE ===")
print((df['UnitPrice'] <= 0).sum())

print("\n=== SAMPLE ROWS ===")
print(df.head(10))


=== DATA TYPES ===
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

=== NULL VALUES PER COLUMN ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133600
Country             0
dtype: int64

=== DUPLICATE ROWS ===
5178

=== CANCELLED ORDERS (InvoiceNo starting with 'C') ===
7856

=== NEGATIVE OR ZERO QUANTITY ===
9192

=== NEGATIVE OR ZERO PRICE ===
2499

=== SAMPLE ROWS ===
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365  

DATA CLEANING

In [4]:
#removing cancelled orders
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print("After removing cancellations:", len(df))

#removing missing CustomerID
df = df.dropna(subset=['CustomerID'])
print("After removing missing CustomerID:", len(df))

#removing negative or zero quantity and price
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
print("Final cleaned rows:", len(df))

#removing duplicates
df = df.drop_duplicates()

#converting CustomerID to int and InvoiceDate to datetime
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')

#creating new column TotalPrice by multiplying Quantity and UnitPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.to_csv("clean_data.csv", index=False)

After removing cancellations: 487622
After removing missing CustomerID: 354345
Final cleaned rows: 354321


EDA

In [5]:
# EDA (on clean data) 

print("=== EDA: BASELINE SNAPSHOT ===")
total_revenue = df['TotalPrice'].sum()
print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Total Customers: {df['CustomerID'].nunique():,}")
print(f"Total Invoices: {df['InvoiceNo'].nunique():,}")
print(f"Total Products: {df['Description'].nunique():,}")
print(f"Date Range: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")

print("\n--- Top 10 products by revenue ---")
top_products = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)
print(top_products)

print("\n--- Monthly revenue trend ---")
monthly = df.set_index('InvoiceDate').resample('ME')['TotalPrice'].sum()
print(monthly)

# Save both for later use in the dashboard
monthly.to_csv("monthly_revenue.csv")
top_products.to_csv("top_products.csv")
print("\nSaved: monthly_revenue.csv, top_products.csv")

=== EDA: BASELINE SNAPSHOT ===
Total Revenue: £7,285,024.64
Total Customers: 3,920
Total Invoices: 16,646
Total Products: 3,844
Date Range: 2010-12-01 to 2011-12-09

--- Top 10 products by revenue ---
Description
PAPER CRAFT , LITTLE BIRDIE           168469.60
REGENCY CAKESTAND 3 TIER              110713.00
WHITE HANGING HEART T-LIGHT HOLDER     94805.50
MEDIUM CERAMIC TOP STORAGE JAR         80291.44
JUMBO BAG RED RETROSPOT                77191.33
PARTY BUNTING                          63050.28
ASSORTED COLOUR BIRD ORNAMENT          52228.68
CHILLI LIGHTS                          45483.91
PAPER CHAIN KIT 50'S CHRISTMAS         40392.08
PICNIC BASKET WICKER 60 PIECES         39619.50
Name: TotalPrice, dtype: float64

--- Monthly revenue trend ---
InvoiceDate
2010-12-31    496477.340
2011-01-31    440876.330
2011-02-28    354618.200
2011-03-31    465784.190
2011-04-30    408733.111
2011-05-31    550359.350
2011-06-30    523775.590
2011-07-31    484545.591
2011-08-31    497194.910
2011-0

Recency: "When did you last visit the shop?"
Frequency: "How many times have you visited?"
Monetary: "How much money have you spent, total?"

In [6]:
# ===== RFM CALCULATION =====

# Set a reference "today" date - one day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate Recency, Frequency, and Monetary value for each customer
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

# Quick look at the RFM table
print("=== RFM TABLE PREVIEW ===")
print(rfm.head(10))

print("\n=== RFM SUMMARY STATISTICS ===")
print(rfm.describe())

# Save it for the clustering step
rfm.to_csv("rfm.csv", index=False)
print("\nSaved: rfm.csv")

=== RFM TABLE PREVIEW ===
   CustomerID  Recency  Frequency  Monetary
0       12346      326          1  77183.60
1       12747        2         11   4196.01
2       12748        1        209  33053.19
3       12749        4          5   4090.88
4       12820        3          4    942.34
5       12821      214          1     92.72
6       12822       71          2    948.88
7       12823       75          5   1759.50
8       12824       60          1    397.12
9       12826        3          7   1474.72

=== RFM SUMMARY STATISTICS ===
         CustomerID      Recency    Frequency       Monetary
count   3920.000000  3920.000000  3920.000000    3920.000000
mean   15562.059694    92.208418     4.246429    1858.424654
std     1576.594671    99.533854     7.199202    7478.631256
min    12346.000000     1.000000     1.000000       3.750000
25%    14208.750000    18.000000     1.000000     298.185000
50%    15569.500000    51.000000     2.000000     644.975000
75%    16913.250000   143.00000

In [7]:
#Libraries for KMeans Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K-MEANS CLUSTERING

In [8]:
# ===== K-MEANS CLUSTERING ON RFM =====

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prepare and scale the RFM features
X = rfm[['Recency','Frequency','Monetary']]
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# Test different numbers of clusters (Elbow + Silhouette)
print("=== FINDING THE RIGHT NUMBER OF SEGMENTS ===")
print(f"{'k':<5}{'WCSS (Elbow)':<18}{'Silhouette Score':<20}")
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    sil = silhouette_score(Xs, km.labels_)
    print(f"{k:<5}{km.inertia_:<18.1f}{sil:<20.4f}")

# Final clustering with chosen k=4 (balances statistical fit + business interpretability)
km_final = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs)
rfm['Cluster'] = km_final.labels_

# Check the actual averages per cluster BEFORE naming them
summary = rfm.groupby('Cluster').agg(
    Customers=('CustomerID','count'),
    AvgRecency=('Recency','mean'),
    AvgFrequency=('Frequency','mean'),
    AvgMonetary=('Monetary','mean'),
    TotalMonetary=('Monetary','sum')
).round(1)
summary['PctCustomers'] = (100*summary['Customers']/summary['Customers'].sum()).round(1)
summary['PctRevenue'] = (100*summary['TotalMonetary']/summary['TotalMonetary'].sum()).round(1)

print("\n=== CLUSTER PROFILES (check these numbers before naming!) ===")
print(summary)

# IMPORTANT: match these cluster IDs to YOUR actual printed numbers above -
# the mapping below matches what we found earlier, but verify your own run first
name_map = {
    3: "Champions",
    1: "Loyal Customers",
    2: "Regular Customers",
    0: "At-Risk / Hibernating"
}
rfm['Segment'] = rfm['Cluster'].map(name_map)
summary['SegmentName'] = summary.index.map(name_map)
summary = summary.sort_values('AvgMonetary', ascending=False)

print("\n=== FINAL NAMED SEGMENTS ===")
for _, row in summary.iterrows():
    print(f"{row['SegmentName']:<22} | {int(row['Customers']):>5} customers ({row['PctCustomers']}%) | "
          f"Recency: {row['AvgRecency']:>6.1f} days | Frequency: {row['AvgFrequency']:>5.1f} | "
          f"Avg Spend: £{row['AvgMonetary']:>10,.2f} | Revenue Share: {row['PctRevenue']}%")

rfm.to_csv("customer_segments.csv", index=False)
summary.to_csv("segment_summary.csv")
print("\nSaved: customer_segments.csv, segment_summary.csv")

=== FINDING THE RIGHT NUMBER OF SEGMENTS ===
k    WCSS (Elbow)      Silhouette Score    
2    8367.4            0.8862              
3    5114.3            0.5810              
4    3789.9            0.5952              
5    2757.6            0.6014              
6    2211.6            0.5968              
7    1805.8            0.5162              
8    1493.6            0.4834              

=== CLUSTER PROFILES (check these numbers before naming!) ===
         Customers  AvgRecency  AvgFrequency  AvgMonetary  TotalMonetary  \
Cluster                                                                    
0              989       244.6           1.6        524.8       518991.4   
1               52        12.2          43.8      31131.7      1618848.3   
2             2876        41.4           4.4       1573.2      4524664.4   
3                3         3.3          36.0     207506.9       622520.6   

         PctCustomers  PctRevenue  
Cluster                            
0          

MARKET BASKET ANALYSIS 

In [9]:
#import sys
#!{sys.executable} -m pip install mlxtend

In [10]:
#Library for Apriori and Association Rules
from mlxtend.frequent_patterns import apriori, association_rules

In [11]:
#Support -> How often the combination appears
#Confidence -> When A is bought, how often is B also bought?
#Lift -> Is A+B happening more than we would expect by chance?

# Lift = 1
# A and B occur together about as much as expected by chance.
# Lift > 1
# A and B occur together more than expected.
# Lift < 1
# A and B occur together less than expected.

In [12]:
# ===== APRIORI - OVERALL MARKET BASKET ANALYSIS =====

# Restrict to top 100 most frequent products to keep computation fast
item_freq = df.groupby('Description')['InvoiceNo'].nunique().sort_values(ascending=False)
top_items = item_freq.head(100).index
df_top = df[df['Description'].isin(top_items)]
print(f"Restricting to top 100 products, covering {df_top['InvoiceNo'].nunique():,} invoices")

# Build the basket matrix: rows = invoices, columns = products, values = bought or not
basket = df_top.groupby(['InvoiceNo','Description'])['Quantity'].sum().unstack().fillna(0)
basket_sets = (basket > 0)
print(f"Basket matrix shape: {basket_sets.shape}")

# Find frequent itemsets and generate rules
frequent_itemsets = apriori(basket_sets, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values('lift', ascending=False)

# Clean up column formatting for readability
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

print(f"\nFrequent itemsets found: {len(frequent_itemsets)}")
print(f"Association rules found: {len(rules)}")

print("\n=== TOP 10 ASSOCIATION RULES BY LIFT ===\n")
for _, r in rules.head(10).iterrows():
    print(f"IF customer buys [{r['antecedents']}]")
    print(f"   THEN they also buy [{r['consequents']}]")
    print(f"   Support={r['support']:.3f}  Confidence={r['confidence']:.2f}  Lift={r['lift']:.2f}\n")

rules.to_csv("association_rules_overall.csv", index=False)
print("Saved: association_rules_overall.csv")

Restricting to top 100 products, covering 13,469 invoices
Basket matrix shape: (13469, 100)

Frequent itemsets found: 162
Association rules found: 132

=== TOP 10 ASSOCIATION RULES BY LIFT ===

IF customer buys [GREEN REGENCY TEACUP AND SAUCER]
   THEN they also buy [ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER]
   Support=0.025  Confidence=0.56  Lift=19.59

IF customer buys [ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER]
   THEN they also buy [GREEN REGENCY TEACUP AND SAUCER]
   Support=0.025  Confidence=0.89  Lift=19.59

IF customer buys [PINK REGENCY TEACUP AND SAUCER]
   THEN they also buy [ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER]
   Support=0.025  Confidence=0.69  Lift=19.57

IF customer buys [ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER]
   THEN they also buy [PINK REGENCY TEACUP AND SAUCER]
   Support=0.025  Confidence=0.72  Lift=19.57

IF customer buys [GREEN REGENCY TEACUP AND SAUCER]
   THEN

In [13]:
# ===== APRIORI - PER SEGMENT (with statistical reliability check) =====

segments = rfm[['CustomerID','Segment']]
df_seg = df.merge(segments, on='CustomerID', how='left')

global_rules = pd.read_csv("association_rules_overall.csv")
global_top_pairs = set(zip(global_rules.head(15)['antecedents'], global_rules.head(15)['consequents']))

MIN_CUSTOMERS_FOR_RELIABLE_APRIORI = 10
all_segment_rules = []

print("=== SEGMENT-WISE APRIORI ===\n")
for seg in df_seg['Segment'].unique():
    seg_df = df_seg[df_seg['Segment'] == seg]
    n_customers = seg_df['CustomerID'].nunique()
    n_invoices = seg_df['InvoiceNo'].nunique()

    print(f"{'='*60}\nSEGMENT: {seg}  ({n_customers} customers, {n_invoices} invoices)\n{'='*60}")

    if n_customers < MIN_CUSTOMERS_FOR_RELIABLE_APRIORI:
        print(f"SKIPPED: only {n_customers} customers - too small a sample for reliable rules.\n")
        continue

    item_freq = seg_df.groupby('Description')['InvoiceNo'].nunique().sort_values(ascending=False)
    top_items = item_freq.head(80).index
    seg_top = seg_df[seg_df['Description'].isin(top_items)]
    basket = seg_top.groupby(['InvoiceNo','Description'])['Quantity'].sum().unstack().fillna(0)
    basket_sets = (basket > 0)

    freq_items = apriori(basket_sets, min_support=0.03, use_colnames=True)
    rules_seg = association_rules(freq_items, metric="lift", min_threshold=1.2)
    rules_seg = rules_seg.sort_values('lift', ascending=False)
    rules_seg['antecedents'] = rules_seg['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules_seg['consequents'] = rules_seg['consequents'].apply(lambda x: ', '.join(list(x)))
    rules_seg['Segment'] = seg
    all_segment_rules.append(rules_seg)

    print(f"Rules found: {len(rules_seg)}")
    for _, r in rules_seg.head(3).iterrows():
        is_unique = (r['antecedents'], r['consequents']) not in global_top_pairs
        tag = " <-- UNIQUE to this segment" if is_unique else " (matches global pattern)"
        print(f"  IF [{r['antecedents']}] THEN [{r['consequents']}]  Lift={r['lift']:.2f}{tag}")
    print()

combined = pd.concat(all_segment_rules, ignore_index=True)
combined.to_csv("association_rules_by_segment.csv", index=False)
print(f"Saved: association_rules_by_segment.csv ({len(combined)} rules)")

=== SEGMENT-WISE APRIORI ===

SEGMENT: At-Risk / Hibernating  (989 customers, 1574 invoices)
Rules found: 36
  IF [PINK REGENCY TEACUP AND SAUCER] THEN [ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER]  Lift=14.52 (matches global pattern)
  IF [ROSES REGENCY TEACUP AND SAUCER , GREEN REGENCY TEACUP AND SAUCER] THEN [PINK REGENCY TEACUP AND SAUCER]  Lift=14.52 (matches global pattern)
  IF [GREEN REGENCY TEACUP AND SAUCER] THEN [ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY TEACUP AND SAUCER]  Lift=13.77 (matches global pattern)

SEGMENT: Regular Customers  (2876 customers, 12687 invoices)
Rules found: 42
  IF [ROSES REGENCY TEACUP AND SAUCER ] THEN [GREEN REGENCY TEACUP AND SAUCER]  Lift=15.91 (matches global pattern)
  IF [GREEN REGENCY TEACUP AND SAUCER] THEN [ROSES REGENCY TEACUP AND SAUCER ]  Lift=15.91 (matches global pattern)
  IF [GARDENERS KNEELING PAD KEEP CALM ] THEN [GARDENERS KNEELING PAD CUP OF TEA ]  Lift=11.90 (matches global pattern)

SEGMENT: Loyal C

In [14]:
# ===== BUSINESS IMPACT - BEFORE vs AFTER =====

summary = pd.read_csv("segment_summary.csv")

# --- Cross-sell opportunity (Apriori-driven) ---
teacups = ['PINK REGENCY TEACUP AND SAUCER', 'GREEN REGENCY TEACUP AND SAUCER', 'ROSES REGENCY TEACUP AND SAUCER']
sub = df[df['Description'].isin(teacups)].copy()
sub['LineRevenue'] = sub['Quantity'] * sub['UnitPrice']
avg_line_revenue = sub['LineRevenue'].mean()

basket = df[df['Description'].isin(teacups)].groupby(['InvoiceNo','Description'])['Quantity'].sum().unstack().fillna(0)
basket_bool = (basket > 0)
exactly_2 = basket_bool[basket_bool.sum(axis=1) == 2]
potential_customers = len(exactly_2)

CONVERSION_ASSUMPTION = 0.25
projected_conversions = int(potential_customers * CONVERSION_ASSUMPTION)
projected_revenue_crosssell = projected_conversions * avg_line_revenue

print("=== CROSS-SELL PROJECTION ===")
print(f"Customers missing 1 of 3 colors: {potential_customers}")
print(f"Assumed {CONVERSION_ASSUMPTION*100:.0f}% conversion: {projected_conversions} customers")
print(f"Projected additional revenue: £{projected_revenue_crosssell:,.2f}")

# --- Win-back opportunity (Clustering-driven) ---
at_risk = summary[summary['SegmentName'] == 'At-Risk / Hibernating'].iloc[0]
regular = summary[summary['SegmentName'] == 'Regular Customers'].iloc[0]

WINBACK_CONVERSION = 0.20
winback_customers = int(at_risk['Customers'] * WINBACK_CONVERSION)
revenue_uplift_per_customer = regular['AvgMonetary'] - at_risk['AvgMonetary']
projected_winback_revenue = winback_customers * revenue_uplift_per_customer

print("\n=== WIN-BACK PROJECTION ===")
print(f"At-Risk customers: {int(at_risk['Customers'])}")
print(f"Assumed {WINBACK_CONVERSION*100:.0f}% conversion: {winback_customers} customers")
print(f"Projected additional revenue: £{projected_winback_revenue:,.2f}")

# --- Combined total ---
total_revenue = df['TotalPrice'].sum()
total_impact = projected_revenue_crosssell + projected_winback_revenue

print("\n=== TOTAL PROJECTED BUSINESS IMPACT ===")
print(f"Current total revenue: £{total_revenue:,.2f}")
print(f"Total projected uplift: £{total_impact:,.2f} ({100*total_impact/total_revenue:.2f}%)")

# Save clean before/after table
before_after = pd.DataFrame([
    {"Area":"Cross-selling (Apriori)",
     "Before":"Teacup sets bought together only by chance; no bundle offer",
     "After": f"{potential_customers} customers targeted with 'complete your set' offer",
     "Projected Revenue Impact": f"£{projected_revenue_crosssell:,.0f}"},
    {"Area":"Customer Retention (K-Means)",
     "Before":"All customers treated identically; At-Risk customers unaddressed",
     "After": f"{int(at_risk['Customers'])} At-Risk customers targeted with win-back offer",
     "Projected Revenue Impact": f"£{projected_winback_revenue:,.0f}"},
    {"Area":"TOTAL",
     "Before":"Reactive, one-size-fits-all marketing",
     "After":"Data-driven, segment + pattern-aware targeting",
     "Projected Revenue Impact": f"£{total_impact:,.0f} (+{100*total_impact/total_revenue:.2f}%)"}
])
before_after.to_csv("before_after_summary.csv", index=False)
print("\nSaved: before_after_summary.csv")

=== CROSS-SELL PROJECTION ===
Customers missing 1 of 3 colors: 404
Assumed 25% conversion: 101 customers
Projected additional revenue: £2,489.58

=== WIN-BACK PROJECTION ===
At-Risk customers: 989
Assumed 20% conversion: 197 customers
Projected additional revenue: £206,534.80

=== TOTAL PROJECTED BUSINESS IMPACT ===
Current total revenue: £7,285,024.64
Total projected uplift: £209,024.38 (2.87%)

Saved: before_after_summary.csv
